In [ ]:
# Setup: Import analytical, plotting, and NLP utilities
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

# Set visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load the preprocessed dataset
data_path = "../data/processed_youtube_data.csv"
df = pd.read_csv(data_path, parse_dates=["trending_date", "publish_time"])
print(f"Loaded {len(df):,} trending records from {data_path}")

# Title & Text Analysis

Video titles represent the primary hook for viewers in YouTube's recommendation feed. In this notebook, we analyze text characteristics of trending titles—including character lengths, word counts, common keywords via word clouds, country-specific linguistic patterns, and the impact of sensationalist elements like ALL-CAPS words and exclamation marks on average views.

In [ ]:
# Title Length Analysis: Compute character length and word count
df["title_length"] = df["title"].astype(str).str.len()
df["title_word_count"] = df["title"].astype(str).apply(lambda x: len(x.split()))

# Calculate correlation between title length and views
corr_length_views = df["title_length"].corr(df["views"])
corr_words_views = df["title_word_count"].corr(df["views"])

print(f"Correlation between Title Length (chars) and Views: {corr_length_views:.4f}")
print(f"Correlation between Title Word Count and Views    : {corr_words_views:.4f}")

# Scatter plot: title_length vs. views
plt.figure(figsize=(10, 6))
plt.scatter(
    df["title_length"],
    df["views"] / 1e6,
    alpha=0.2,
    color="#1f77b4",
    edgecolors="none",
    s=20,
)

plt.title("Title Length (Characters) vs. View Count", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Title Character Length", fontsize=12, labelpad=8)
plt.ylabel("Views (Millions)", fontsize=12, labelpad=8)
plt.grid(True, linestyle="--", alpha=0.6)

# Overlay mean views per title length bin for clearer trend visibility
length_bins = pd.cut(df["title_length"], bins=range(0, 110, 5))
binned_means = df.groupby(length_bins, observed=False)["views"].mean() / 1e6
bin_centers = [interval.mid for interval in binned_means.index]
plt.plot(bin_centers, binned_means.values, color="#e74c3c", linewidth=2.5, marker="o", label="Binned Mean Views")

plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.savefig("../outputs/title_length_vs_views.png", dpi=300, bbox_inches="tight")
plt.show()

## Word Cloud of Trending Titles

To identify overarching themes across all trending videos, we combine and preprocess titles by stripping special characters, lowercasing text, and filtering standard English stopwords.

In [ ]:
# Word Cloud: Aggregate and clean titles across all trending videos
all_titles_text = " ".join(df["title"].dropna().astype(str))

# Clean text: retain only alphabetic characters and lowercase
cleaned_titles_text = re.sub(r"[^a-zA-Z\s]", " ", all_titles_text).lower()

# Define stopwords: combine WordCloud STOPWORDS with YouTube metadata terms
custom_stopwords = set(STOPWORDS)
custom_stopwords.update([
    "video", "official", "trailer", "vs", "ft", "feat", "full", "episode", "part",
    "the", "a", "an", "is", "in", "on", "of", "to", "and", "with", "for", "by"
])

# Generate WordCloud visualization (figsize=(15, 8), white background, max 100 words)
wordcloud_all = WordCloud(
    width=1500,
    height=800,
    background_color="white",
    max_words=100,
    stopwords=custom_stopwords,
    colormap="viridis",
    random_state=42,
).generate(cleaned_titles_text)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud_all, interpolation="bilinear")
plt.axis("off")
plt.title("Top 100 Keywords in Trending YouTube Titles", fontsize=18, fontweight="bold", pad=15)
plt.tight_layout(pad=0)
plt.savefig("../outputs/wordcloud_global.png", dpi=300, bbox_inches="tight")
plt.show()

## Word Clouds by Country

Examining titles partitioned by country highlights geographical interests, entertainment formats, and cultural trends unique to the United States (US), India (IN), and Great Britain (GB).

In [ ]:
# Country-wise Word Clouds: 3 subplots (1 row, 3 columns) for US, IN, and GB
countries = ["US", "IN", "GB"]
colormaps = {"US": "Blues_r", "IN": "YlOrRd_r", "GB": "PuRd_r"}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, country in zip(axes, countries):
    # Filter and extract country titles
    country_subset = df[df["country"] == country]["title"].dropna().astype(str)
    country_text = " ".join(country_subset)
    cleaned_country_text = re.sub(r"[^a-zA-Z\s]", " ", country_text).lower()

    # Generate country-specific word cloud
    wc_country = WordCloud(
        width=600,
        height=400,
        background_color="white",
        max_words=80,
        stopwords=custom_stopwords,
        colormap=colormaps.get(country, "viridis"),
        random_state=42,
    ).generate(cleaned_country_text)

    ax.imshow(wc_country, interpolation="bilinear")
    ax.set_title(f"Trending Titles: {country}", fontsize=15, fontweight="bold", pad=10)
    ax.axis("off")

plt.suptitle("Trending Title Keywords by Region (US vs. IN vs. GB)", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../outputs/wordcloud_by_country.png", dpi=300, bbox_inches="tight")
plt.show()

## Capital Letters & Punctuation Analysis

Content creators often leverage formatting techniques like ALL-CAPS words or exclamation marks (`!`) to draw attention and convey urgency. Here, we evaluate whether these tactics correlate with higher average viewership.

In [ ]:
# Caps and Punctuation Analysis
# 1. has_caps_words: True if title contains any all-caps word (length > 2 to avoid single/double letter acronyms)
df["has_caps_words"] = df["title"].apply(
    lambda t: bool(re.search(r"\b[A-Z]{3,}\b", str(t)))
)

# 2. has_exclamation: True if title contains '!'
df["has_exclamation"] = df["title"].astype(str).str.contains("!", regex=False)

# Calculate mean views for both features
caps_stats = df.groupby("has_caps_words")["views"].agg(["mean", "count"]).rename(index={False: "Without ALL-CAPS", True: "With ALL-CAPS"})
excl_stats = df.groupby("has_exclamation")["views"].agg(["mean", "count"]).rename(index={False: "Without '!'", True: "With '!'"})

print("Mean Views by ALL-CAPS Presence:")
display(caps_stats)
print("\nMean Views by Exclamation Mark Presence:")
display(excl_stats)

# Create comparison bar charts side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: ALL-CAPS comparison
bars1 = ax1.bar(
    caps_stats.index,
    caps_stats["mean"] / 1e6,
    color=["#3498db", "#e74c3c"],
    width=0.5,
)
ax1.set_title("Average Views: ALL-CAPS Words in Title", fontsize=13, fontweight="bold", pad=12)
ax1.set_ylabel("Average Views (Millions)", fontsize=11)
ax1.set_ylim(0, (caps_stats["mean"].max() / 1e6) * 1.18)

for bar in bars1:
    h = bar.get_height()
    ax1.annotate(
        f"{h:.2f}M",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontweight="semibold",
    )

# Chart 2: Exclamation Mark comparison
bars2 = ax2.bar(
    excl_stats.index,
    excl_stats["mean"] / 1e6,
    color=["#2ecc71", "#f39c12"],
    width=0.5,
)
ax2.set_title("Average Views: Exclamation Mark '!' in Title", fontsize=13, fontweight="bold", pad=12)
ax2.set_ylabel("Average Views (Millions)", fontsize=11)
ax2.set_ylim(0, (excl_stats["mean"].max() / 1e6) * 1.18)

for bar in bars2:
    h = bar.get_height()
    ax2.annotate(
        f"{h:.2f}M",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontweight="semibold",
    )

plt.tight_layout()
plt.savefig("../outputs/caps_exclamation_comparison.png", dpi=300, bbox_inches="tight")
plt.show()